In [1]:
import torch
import torch.nn as nn

# A simple model
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

In [2]:
# Create optimizer — pass it model.parameters() so it knows what to update
# lr = learning rate — how big each step is
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- One training step ---

In [4]:
# 1. Generate a fake batch
x      = torch.rand(5, 4)          # 5 samples, 4 features
target = torch.rand(5, 1)          # 5 continuous targets (regression)

loss_fn = nn.MSELoss()

In [6]:
# 2. Forward pass — get predictions
predictions = model(x)

In [7]:
# 3. Compute loss
loss = loss_fn(predictions, target)
print(loss.item())    # .item() converts tensor to a plain Python float

0.4015101492404938


In [8]:
# 4. Zero gradients from previous step (remember the accumulation gotcha!)
optimizer.zero_grad()

In [9]:
# 5. Backward pass — compute gradients
loss.backward()

In [10]:
# 6. Optimizer steps — updates every weight using its gradient
optimizer.step()

## Watching a model actually learn

In [15]:
import torch
import torch.nn as nn

# True relationship we want to learn: y = 2x + 1
# We'll generate data from this and see if the model discovers it

torch.manual_seed(42)   # for reproducibility

In [16]:
# Data: 100 samples
x = torch.rand(100, 1) * 10        # inputs between 0 and 10
y = 2 * x + 1 + torch.randn(100, 1) * 0.5   # y = 2x+1, plus a little noise

# The simplest model possible — one linear layer (linear regression!)
model   = nn.Linear(1, 1)          # 1 input, 1 output
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

In [17]:
# Train for 100 steps
for step in range(100):
    predictions = model(x)              # forward pass
    loss        = loss_fn(predictions, y)   # compute loss

    optimizer.zero_grad()               # zero gradients
    loss.backward()                     # compute gradients
    optimizer.step()                    # update weights

    if step % 10 == 0:
        print(f"Step {step:3d} | Loss: {loss.item():.4f}")

Step   0 | Loss: 146.1965
Step  10 | Loss: 30.0477
Step  20 | Loss: 0.2941
Step  30 | Loss: 3.5122
Step  40 | Loss: 1.7064
Step  50 | Loss: 0.1953
Step  60 | Loss: 0.4119
Step  70 | Loss: 0.2192
Step  80 | Loss: 0.1984
Step  90 | Loss: 0.1916


In [18]:
# After training, see what the model learned
print(f"\nLearned weight: {model.weight.item():.4f}")   # should be ≈ 2.0
print(f"Learned bias:   {model.bias.item():.4f}")       # should be ≈ 1.0


Learned weight: 1.9368
Learned bias:   1.3545


In [19]:
# Too large — loss bounces around or explodes
optimizer = torch.optim.Adam(model.parameters(), lr=1.0)

# Too small — loss decreases but painfully slowly
optimizer = torch.optim.Adam(model.parameters(), lr=0.000001)

# Sweet spot for Adam — usually start here
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Rule of thumb: Start with lr=0.001 for Adam. If loss barely moves, try 0.01. If loss explodes or bounces, try 0.0001. You'll develop a feel for this quickly.

In [21]:
# SGD — you control everything, but needs more tuning
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
#                                                         ↑ momentum helps
#                                                           escape flat regions

# Adam — handles most of that automatically
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)